# HydraY NNUE — la rete a 1024 con il DOPPIO del budget di training

Runtime → Cambia tipo di runtime → **GPU (T4)**, poi Runtime → **Esegui tutte**.
Durata ~3h. **Non lasciare la scheda inattiva.**

### A cosa serve
Identico in tutto alla rete appena spedita con la 3.1.0 — stesso dataset, stessi
4 king bucket specchiati, stessi 8 bucket di uscita, stesso WDL, stesso LR
iniziale. L'unica variabile e' il **budget**: 80 superbatch invece di 40.

### Perche'
La rete a 1024 ha ricevuto esattamente lo schedule della 512: 40 superbatch. Ha
il doppio dei parametri e ha avuto lo stesso tempo per addestrarli.

Questo e' gia' successo una volta. HalfKA a 20 superbatch **perdeva** 11,3 Elo
contro la 512; a 40 vinceva 26,5. Il budget era la variabile nascosta, ed era
invisibile finche' non e' stata mossa. Una rete grande sotto-addestrata sembra
identica a una rete grande inutile.

### La cosa importante: questo run risponde a DUE domande
80 superbatch sono 8,0 miliardi di campioni su 1,25 miliardi di posizioni, cioe'
**6,4 epoche**. A quel punto la validation loss diventa informativa:

- se scende fino in fondo → il tetto era il **budget**, e la rete e' migliore;
- se risale mentre la training loss continua a scendere → sei limitato dai
  **dati**, non dal budget, e il passo successivo e' il dataset da 2,7B.

Perche' funzioni, pero', il validation set deve essere **davvero** escluso dal
training — e nel notebook precedente non lo era (la fetta di coda restava dentro
`data.bin`). E' l'unica differenza sostanziale rispetto a quel notebook, ed e'
in fondo il motivo per cui questo run vale la pena: vedi la cella del
validation set.

### Il prezzo di NPS e' gia' scontato
Non cambia nulla rispetto alla 3.1.0: stessa architettura, stesso costo per
nodo. Lo SPRT sara' testa a testa contro la rete spedita, quindi misura il
budget in purezza.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    # L'output va riletto e ristampato da Python: subprocess.run() senza capture
    # scrive sui file descriptor del KERNEL, che Colab non mostra nella cella.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + dataset (le STESSE due parti della rete spedita) ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]
P1, P2 = find('hydray_v6_part1.bin.zst'), find('hydray_v6_part2.bin.zst')
os.environ['P1'], os.environ['P2'] = P1, P2
print('parte 1:', P1, os.path.getsize(P1), 'byte')
print('parte 2:', P2, os.path.getsize(P2), 'byte')

NET_ID   = 'hydray-1024-80sb'
TOTAL_SB = 80          # il doppio della rete spedita: e' l'unica variabile
STAGE1   = TOTAL_SB // 2
TRAINER  = '/content/th/nnue/trainer'

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
#
# NOTA: si clona ancora il branch `nnue-1024`, non `dev`. I commit della 3.1.0
# esistono solo in locale finche' non vengono pushati, quindi su GitHub `dev` e'
# ancora a 512 neuroni. Il trainer sui due branch e' identico.
sh('rm -rf /content/th')
sh('git clone --depth 1 --branch nnue-1024 https://github.com/ThomasGhione/HydraY /content/th')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 1024;' in src, 'NON e il branch a 1024 neuroni'
assert 'const INPUT_BUCKETS: usize = 4;' in src, 'i king bucket devono restare 4'
tr = open(f'{TRAINER}/src/bin/trainer.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer.rs non e a 1024'
print('branch nnue-1024, 1024 neuroni, 4 king bucket: ok')

In [ ]:
# --- decompressione in due parti (ogni zstd esce e libera la cache di Drive) ---
sh('apt-get -qq install -y zstd >/dev/null')

TOT, HALF = 41070409664, 20535204832
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~50 GiB')
assert free_gb > 52, 'disco insufficiente'

sh('zstd -d -T0 --long=27 -c "$P1" > /content/data.bin')
assert os.path.getsize('/content/data.bin') == HALF, 'parte 1 di taglia inattesa'
print('parte 1 ok'); sh('df -h /content | tail -1')

sh('zstd -d -T0 --long=27 -c "$P2" >> /content/data.bin')
SIZE = os.path.getsize('/content/data.bin')
print('data.bin:', SIZE, 'byte =', SIZE // 32, 'posizioni')
assert SIZE == TOT, f'taglia finale inattesa: {SIZE}'
sh('df -h /content | tail -1')

In [ ]:
# --- validation set: 1000 MiB RITAGLIATI VIA dal training ---
# Il notebook precedente copiava la coda in test.bin e la LASCIAVA dentro
# data.bin. Quella "validation" loss misurava la rete su dati che aveva gia'
# visto: puo' solo scendere, e quindi non puo' mostrare overfitting. Per un run
# che serve proprio a distinguere "poco budget" da "pochi dati", sarebbe stata
# inutile — e ingannevole, perche' sembra una misura buona.
#
# Qui la coda viene copiata e poi TRONCATA dal file di training, quindi e' un
# vero held-out. Costa il 2,5% delle posizioni.
# 1 MiB = 32768 record da 32 byte: i confini in MiB sono allineati ai record.
TEST_MIB = 1000
skip_mib = SIZE // (1024*1024) - TEST_MIB
sh(f'dd if=/content/data.bin bs=1M skip={skip_mib} count={TEST_MIB} of=/content/test.bin status=progress')
assert os.path.getsize('/content/test.bin') == TEST_MIB * 1024*1024

os.truncate('/content/data.bin', skip_mib * 1024*1024)
TRAIN_SIZE = os.path.getsize('/content/data.bin')
# data.bin non e' un multiplo esatto di MiB: l'ultimo MiB parziale (< 1 MiB)
# cade fuori sia dal training sia dall'held-out, ed e' irrilevante.
assert TRAIN_SIZE % 32 == 0 and TRAIN_SIZE == skip_mib * 1024*1024
print(f'training: {TRAIN_SIZE//32} posizioni   held-out: {TEST_MIB*1024*1024//32} posizioni')
print(f'{TOTAL_SB} superbatch = {TOTAL_SB*100_007_936/(TRAIN_SIZE//32):.1f} epoche')

In [ ]:
# --- training, tappa 1 di 2: superbatch 1-40 ---
# Diviso in due tappe SOLO per sopravvivenza: se la sessione Colab muore a
# meta', il checkpoint della tappa 1 e' su Drive e si riparte da li' invece di
# buttare 3 ore. Il dataset e' lo stesso in entrambe le tappe (nessuna fetta da
# scambiare) e lo schedule del learning rate keya su TOTAL_SB, quindi il calo
# resta dove sarebbe stato in un run unico.
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda '
   f'TEST_PATH=/content/test.bin STAGE_END={STAGE1} '
   f'cargo run -r --bin trainer --features cuda -- '
   f'/content/data.bin {TOTAL_SB} {NET_ID}')

CKPT1 = f'{TRAINER}/checkpoints/{NET_ID}-{STAGE1}'
assert os.path.isdir(CKPT1), f'checkpoint della tappa 1 assente: {CKPT1}'
sh(f'cp -r {CKPT1} /content/drive/MyDrive/')
print(f'tappa 1 finita, checkpoint al superbatch {STAGE1} salvato su Drive')

In [ ]:
# --- training, tappa 2 di 2: superbatch 41-80 ---
# load_from_checkpoint ripristina anche lo stato dell'optimiser, quindi i momenti
# di AdamW attraversano il confine invece di ripartire da freddo.
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda '
   f'TEST_PATH=/content/test.bin STAGE_END={TOTAL_SB} '
   f'cargo run -r --bin trainer --features cuda -- '
   f'/content/data.bin {TOTAL_SB} {NET_ID} {STAGE1+1} checkpoints/{NET_ID}-{STAGE1}')

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# payload 6.326.288 + padding a 64 byte. La rete a 512 pesa 3.163.200: la
# taglia e' il controllo piu' rapido che l'architettura sia quella giusta.
assert 6326288 <= sz < 6326288 + 64, f'taglia {sz}: NON e la rete a 1024'
print('quantised.bin:', sz, 'byte — 1024 neuroni confermati\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*66)
print('RIFERIMENTO — la rete spedita con la 3.1.0 (stessa architettura, 40 SB),')
print('misurata sul binario di casa con lo stesso sanity.rs:')
print('  startpos            71      middlegame ~24 pezzi   941')
print('  KQvK               646      KRPvKR                  72')
print('  su un cavallo      792      re attivi (finale)      31')
print('  arroccati corto   1727      training loss      0.012920')
print()
print('Riporta: training loss finale, VALIDATION loss (e il suo andamento negli')
print('ultimi superbatch: scende ancora o ha girato?), e i sanity qui sopra.')
print('='*66)

## Come leggere il risultato

### 1. Prima di tutto, la validation loss
E' la ragione per cui questo run esiste, ed e' informativa solo perche' la fetta
di held-out e' stata tolta dal training.

- **scende fino all'ultimo superbatch** → il tetto era il budget. La rete e'
  probabilmente migliore, e vale la pena chiedersi se anche 120 SB rendano;
- **ha girato verso l'alto** mentre la training loss continuava a scendere →
  overfitting: sei limitato dai **dati**, non dal budget. Il prossimo passo
  diventa il dataset da 2,7B, e la conclusione di A5 ("piu' dati non valgono
  niente") va considerata annullata, perche' era misurata su una 512 satura.

Attenzione: la validation loss dice **da che parte guardare dopo**, non se la
rete e' piu' forte. Quello lo dice solo lo SPRT.

### 2. I sanity, in un minuto
Confronto contro i valori della rete spedita stampati dalla cella sopra. Se
KQvK e mediogioco crollano, la rete e' peggiore e lo sai prima di spendere ore
di partite.

Il valore da guardare con piu' attenzione e' **KQvK: 646 cp contro un vero di
circa 900**. Se il budget in piu' serve a qualcosa, e' plausibile che si veda
prima li' che altrove — quella regione ha dati freschi (i due batch di finali)
che 40 superbatch potrebbero non aver finito di assorbire.

### 3. Poi lo SPRT, testa a testa
Contro la rete della 3.1.0, non contro una baseline comune: due misure separate
sommano i loro errori, il testa a testa no.

Un avvertimento sul controllo di tempo. La 1024 contro la 512 valeva +2,71 a
4+0.04 e +10,69 a 10+0.1: lo stesso cambiamento valeva quattro volte tanto al
controllo lungo. Qui architettura e NPS sono identici, quindi la distorsione
dovrebbe sparire — ma se il risultato a 4+0.04 esce piatto, prima di archiviarlo
vale la pena rifarlo a 10+0.1.
